HyDRA_Magnetic-1.ipynb.txt

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/11cQ8MLeVcJRvN9jh9XbNxKOuuBNo6HXN

# HyDRA-Competitive
### Target: PPL ≤ GPT-2-small (~35) on WikiText-2

**Architecture:** 12L × 512d × 8H · 64M params · seq=1024  
**Data:** OpenWebText (~38GB) — same distribution as GPT-2 training  
**Loss:** CE dominant (λ=1.0) + minimal KL (λ=0.1) + geometry reg (λ=0.05)  
**Key fix:** AngularLMHeadV2, temperature frozen=20.0, no hidden-state dominance

| | GPT-2-small | HyDRA-Competitive |
|---|---|---|
| Params | 117M | 64M |
| Data | 40GB WebText | 38GB OpenWebText |
| Seq len | 1024 | 1024 |
| Target PPL | 35 | ≤ 35 |

> **Run order:** Cell 1 (env) → 2 (src) → 3 (CFG) → 4 (models) → 5 (data) → 6 (train) → 7 (chat)

Cell 1 — Environment Setup
==========================
Installs dependencies, optionally mounts Google Drive, creates the output
directory tree, and sets global determinism seeds.

Storage strategy (automatic):
  • Google Drive available  → PAPER_ROOT = /content/drive/MyDrive/HydraPaper
  • Drive unavailable/skipped → PAPER_ROOT = /content/HydraPaper
    (persists for the Colab session; download artifacts via Cell 12 ZIP export)

Directory structure:
    /checkpoints/   model snapshots (.pt)
    /logs/          raw training logs (.csv)
    /results/       derived metrics
    /figures/       publication figures (PNG 300 DPI + PDF vector)
    /tables/        LaTeX tables (booktabs format)
    /configs/       JSON reproducibility manifests
    /exports/       final ZIP archive

In [1]:
import subprocess, sys, os

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[WARN] {cmd!r} → {r.stderr[:200]}')

print('Installing dependencies...')
_run('pip install torch tqdm transformers accelerate datasets POT --quiet')
print('✅ Dependencies installed')

# ── Google Drive (optional) ────────────────────────────────
from pathlib import Path

USE_DRIVE = True   # FIX: try Drive first; falls back to local if unavailable
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PAPER_ROOT = Path('/content/drive/MyDrive/HydraPaper_VariantF').resolve()
    USE_DRIVE  = True
    print('✅ Google Drive mounted')
except Exception as _drive_err:
    PAPER_ROOT = Path('/content/HydraPaper_VariantF')
    print(f'⚠️  Drive unavailable ({type(_drive_err).__name__}: {_drive_err})')
    print(f'   → Using local storage: {PAPER_ROOT}')
    print('   Artifacts will be in the session ZIP export (Cell 12).')

SUBDIRS = ['checkpoints','logs','results','figures','tables','configs','exports']
for d in SUBDIRS:
    (PAPER_ROOT / d).mkdir(parents=True, exist_ok=True)
print(f'✅ Output root: {PAPER_ROOT}  (drive={USE_DRIVE})')

# ── Determinism ────────────────────────────────────────────
import torch, random, numpy as np
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False   # precision guard for Lorentz ops
torch.backends.cudnn.allow_tf32       = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Seeds set (seed={SEED})  device={DEVICE}  torch={torch.__version__}')
print(f'📂 PAPER_ROOT absoluto: {PAPER_ROOT}')
os.chdir(PAPER_ROOT)
print(f'📍 Working dir: {Path.cwd()}')

Installing dependencies...
✅ Dependencies installed
Mounted at /content/drive
✅ Google Drive mounted
✅ Output root: /content/drive/MyDrive/HydraPaper_VariantF  (drive=True)
✅ Seeds set (seed=42)  device=cuda  torch=2.10.0+cu128
📂 PAPER_ROOT absoluto: /content/drive/MyDrive/HydraPaper_VariantF
📍 Working dir: /content/drive/MyDrive/HydraPaper_VariantF


Cell 2 — Source Upload & Mount  [HARDENED: cache-purge + version lock]
=======================================================================
Purges any cached cgt installation, extracts the uploaded zip fresh,
then PROVES which code version is active before allowing execution to
continue.  If target_radius != 1.5, execution STOPS here.

In [2]:
import importlib, sys, os, zipfile

# ── STEP 0: Hard purge of all cached cgt modules ───────────
# Colab may hold stale compiled .pyc or already-imported modules in memory.
# Remove them ALL before touching the file system.
_stale = [k for k in sys.modules if k == 'cgt' or k.startswith('cgt.')]
for _k in _stale:
    del sys.modules[_k]
print(f"Purged {len(_stale)} cached cgt module(s) from sys.modules")

# ── STEP 1: Wipe previous extraction on disk ───────────────
import shutil
for _old in ['/content/src', '/content/cgt']:
    if os.path.exists(_old):
        shutil.rmtree(_old)
        print(f"Deleted old directory: {_old}")

# ── STEP 2: Upload and extract ─────────────────────────────
from google.colab import files as _colab_files
print('Upload src_FULL_V5.zip when the dialog appears...')
_uploaded = _colab_files.upload(target_dir='/tmp')
if not _uploaded:
    raise RuntimeError('No file uploaded.')
_zip_name = list(_uploaded.keys())[0]
_zip_path = f'/{_zip_name}'

os.makedirs('/content/src', exist_ok=True)
with zipfile.ZipFile(_zip_path, 'r') as _zf:
    _zf.extractall('/content/src')
print(f'Extracted: {_zip_name}')

# ── STEP 3: Mount sys.path ─────────────────────────────────
_candidates = ['/content/src', '/content/src/src', '/content']
_cgt_root = next(
    (p for p in _candidates if os.path.isdir(os.path.join(p, 'cgt'))), None
)
if _cgt_root is None:
    raise RuntimeError('cgt/ not found in extracted zip. Wrong zip uploaded?')
if _cgt_root not in sys.path:
    sys.path.insert(0, _cgt_root)

# ── STEP 4: Grep the actual file on disk (mandatory) ───────
import subprocess
_distill_path = os.path.join(_cgt_root, 'cgt', 'distillation', 'distillation_v2.py')
if not os.path.exists(_distill_path):
    raise FileNotFoundError(f'distillation_v2.py not found at {_distill_path}')

_grep = subprocess.run(
    ['grep', '-n', 'target_radius', _distill_path],
    capture_output=True, text=True
)
print("\n[DISK VERIFY] grep target_radius distillation_v2.py:")
print(_grep.stdout.strip())

# ── STEP 5: Parse the exact value from file ────────────────
_target_on_disk = None
for _line in _grep.stdout.splitlines():
    if '= ' in _line and 'target_radius' in _line and '#' not in _line.split('target_radius')[0]:
        try:
            _target_on_disk = float(_line.split('=')[1].split('#')[0].strip())
            break
        except ValueError:
            pass

print(f"[DISK VERIFY] target_radius on disk = {_target_on_disk}")

# ── STEP 6: FAIL-FAST if wrong version ─────────────────────
if _target_on_disk is None:
    raise RuntimeError(
        "ABORT: Could not parse target_radius from distillation_v2.py. "
        "Inspect the file manually."
    )
if _target_on_disk >= 2.0:
    raise RuntimeError(
        f"ABORT: target_radius={_target_on_disk} on disk — OLD code loaded. "
        f"Upload src_FULL_V4.zip (not src__7_.zip or any original version). "
        f"Expected: target_radius=1.5"
    )

print(f"\n✅ DISK VERSION CONFIRMED: target_radius={_target_on_disk}")

# ── STEP 7: Import and runtime confirmation ────────────────
import cgt
from cgt.distillation.distillation_v2 import TeacherDistillationLossV2
import inspect, re

_src = inspect.getsource(TeacherDistillationLossV2._radius_loss)
_m = re.search(r'target_radius\s*=\s*([\d.]+)', _src)
_runtime_val = float(_m.group(1)) if _m else None

print(f"[RUNTIME VERIFY] target_radius in loaded class = {_runtime_val}")

if _runtime_val is None or _runtime_val >= 2.0:
    raise RuntimeError(
        f"ABORT: Runtime target_radius={_runtime_val} — module reload failed. "
        f"Kernel may be caching a stale version. "
        f"Go to Runtime → Restart runtime, then re-run from Cell 1."
    )

print(f"✅ RUNTIME VERSION CONFIRMED: target_radius={_runtime_val}")
print(f"✅ cgt mounted  root={_cgt_root}  version={cgt.__version__}")

# ── STEP 8: Verify V4 new modules are present ──────────────
_v4_checks = {
    'HyperbolicProjectorV3': 'cgt.distillation.hyperbolic_projector',
    'GeodesicLMHeadV2':      'cgt.models.geodesic_lm_head',
    'RiemannianAdamW':       'cgt.dynamics.riemannian_adamw',
}
_v4_missing = []
for _cls, _mod in _v4_checks.items():
    try:
        _m = importlib.import_module(_mod)
        getattr(_m, _cls)
        print(f"  ✅ {_cls} present in {_mod}")
    except Exception as _e:
        _v4_missing.append(f"{_cls}: {_e}")
        print(f"  ⚠️  {_cls} missing — {_e}")

if _v4_missing:
    print(f"⚠️  V4 modules missing (non-critical, training still works): {_v4_missing}")
else:
    print("✅ src_FULL_V4.zip confirmed — all new modules present")
print("\n" + "="*60)
print("V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.")
print("="*60)

Purged 0 cached cgt module(s) from sys.modules
Upload src_FULL_V5.zip when the dialog appears...


Saving src.zip to /tmp/src.zip
Extracted: /tmp/src.zip

[DISK VERIFY] grep target_radius distillation_v2.py:
942:        # FIX: target_radius 4.0 → 1.5
961:        target_radius = 1.5
962:        return F.mse_loss(radius.clamp(max=4.0), torch.full_like(radius, target_radius))
[DISK VERIFY] target_radius on disk = 1.5

✅ DISK VERSION CONFIRMED: target_radius=1.5
[RUNTIME VERIFY] target_radius in loaded class = 1.5
✅ RUNTIME VERSION CONFIRMED: target_radius=1.5
✅ cgt mounted  root=/content/src/src  version=1.0.0
  ✅ HyperbolicProjectorV3 present in cgt.distillation.hyperbolic_projector
  ✅ GeodesicLMHeadV2 present in cgt.models.geodesic_lm_head
  ✅ RiemannianAdamW present in cgt.dynamics.riemannian_adamw
✅ src_FULL_V4.zip confirmed — all new modules present

V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.


Cell 3 — HyDRA-Competitive Configuration

In [3]:
import yaml
from pathlib import Path

PAPER_ROOT = Path("/content/drive/MyDrive/HydraPaper_Magnetic")
SEQ_LEN    = 512

CFG = {
    "model": {
        "vocab_size":        50257,
        "n_embd":            512,    # 12L×512d = 64.4M params
        "n_layer":           12,
        "n_head":            8,
        "n_positions":       SEQ_LEN,
        "hyperbolic_dim":    512,
        "initial_curvature": 1.0,
        "use_dynamics":      True,   # ← Magnetic: enable Kuramoto dynamics
    },
    "dynamics": {
        "coupling_strength": 0.3,   # ← Magnetic: Kuramoto coupling K
        "dt":                0.05,   # max allowed by precision guard
        "num_steps":         1,
    },
    "training": {
        "alpha":             1.0,    # CE weight — language dominates
        "temperature":       1.0,
        "lambda_distill":    0.01,    # minimal KL from teacher
        "lambda_hidden":     0.05,   # Gemini: boost hidden transfer
        "lambda_radius":     0.05,   # geometry anchor
        "lambda_contrast":   0.05,    # off — AngularHead handles this
        "label_smoothing":   0.1,
        "learning_rate":     3e-4,   # cosine decay over 100k steps
        "weight_decay":      0.1,
        "max_steps":         200_000,  # Gemini: extend training horizon
        "warmup_steps":      2000,
        "gradient_clip":     1.0,
        "eval_every":        1000,
        "log_every":         100,
        "checkpoint_every":  5000,
        "lr_floor":          0.05,   # Gemini: SGDR needs lower floor
        "teacher_hidden_dim":768,
        "batch_size":        8,      # seq=512 → 8×512=4096 tokens/step
        "dataset":           "openwebtext",
    },
    "early_stopping": {
        "patience":          20,
        "min_delta":         0.005,
        "ema_beta":          0.9,
        "ema_beta_fast":     0.3,
        "min_steps":         20000,  # no early stop before 20k
        "window_size":       5,
        "noise_mult":        2.0,
        "plateau_threshold": 10,
        "logit_std_delta":   0.1,
    },
}

print("HyDRA-Competitive config:")
print(f"  model      : {CFG["model"]["n_layer"]}L × {CFG["model"]["n_embd"]}d × {CFG["model"]["n_head"]}H")
print(f"  seq_len    : {SEQ_LEN}")
print(f"  max_steps  : {CFG["training"]["max_steps"]:,}")
print(f"  batch_size : {CFG["training"]["batch_size"]} × {SEQ_LEN} = {CFG["training"]["batch_size"]*SEQ_LEN:,} tokens/step")
print(f"  total tokens: {CFG["training"]["max_steps"]*CFG["training"]["batch_size"]*SEQ_LEN/1e9:.1f}B")
print(f"  λ_CE={CFG["training"]["alpha"]}  λ_KL={CFG["training"]["lambda_distill"]}  λ_hid={CFG["training"]["lambda_hidden"]}")
print("✅ CFG ready")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {torch.cuda.get_device_name(0)}  {total:.1f}GB")
    if total < 14:
        print("  ⚠️  < 14GB — reduce batch_size to 4 or n_layer to 8")
    else:
        print("  ✅ sufficient memory for 12L×512d")

HyDRA-Competitive config:
  model      : 12L × 512d × 8H
  seq_len    : 512
  max_steps  : 200,000
  batch_size : 8 × 512 = 4,096 tokens/step
  total tokens: 0.8B
  λ_CE=1.0  λ_KL=0.01  λ_hid=0.05
✅ CFG ready
DEVICE: cuda
  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  102.0GB
  ✅ sufficient memory for 12L×512d


Cell 5 — Teacher + Student

In [4]:
import time
from transformers import GPT2Tokenizer
from cgt.api.entrypoint import SafeHyperbolicModel, SafeModelConfig
from cgt.distillation import GPT2TeacherWrapperV2
from cgt.models.geodesic_lm_head import AngularLMHeadV2
from cgt.models.sublattice_lm_head import SublatticeLMHead

# ── Tokenizer ─────────────────────────────────────────────────────────────
_tok_cache = PAPER_ROOT / "models" / "gpt2_cache"
if _tok_cache.exists():
    tokenizer = GPT2Tokenizer.from_pretrained(str(_tok_cache))
    print(f"Tokenizer: loaded from Drive cache")
else:
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.save_pretrained(str(_tok_cache))
    print(f"Tokenizer: downloaded + cached to Drive")
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer: vocab={VOCAB_SIZE}")

# ── Teacher ───────────────────────────────────────────────────────────────
_model_drive = PAPER_ROOT / "models" / "gpt2_model_cache"
if _model_drive.exists() and any(_model_drive.iterdir()):
    print("Teacher: loading from Drive cache...")
    teacher = GPT2TeacherWrapperV2(model_name=str(_model_drive), device=DEVICE)
    print(f"Teacher: GPT-2-small {teacher.config.n_embd}d [frozen] — from cache")
else:
    print("Teacher: downloading GPT-2-small (~548MB)...")
    teacher = GPT2TeacherWrapperV2(model_name="gpt2", device=DEVICE)
    _model_drive.mkdir(parents=True, exist_ok=True)
    teacher.model.save_pretrained(str(_model_drive))
    print(f"Teacher: GPT-2-small {teacher.config.n_embd}d [frozen] — saved to Drive")

# ── Student ───────────────────────────────────────────────────────────────
torch.manual_seed(42)
student_cfg = SafeModelConfig(
    vocab_size        = VOCAB_SIZE,
    n_embd            = CFG["model"]["n_embd"],
    n_layer           = CFG["model"]["n_layer"],
    n_head            = CFG["model"]["n_head"],
    n_positions       = CFG["model"]["n_positions"],
    initial_curvature = CFG["model"]["initial_curvature"],
    use_dynamics      = CFG["model"]["use_dynamics"],
    hyperbolic_dim    = CFG["model"]["hyperbolic_dim"],
    coupling_strength = CFG["dynamics"]["coupling_strength"],
    dt                = CFG["dynamics"]["dt"],
    num_steps         = CFG["dynamics"]["num_steps"],
)
student = SafeHyperbolicModel(student_cfg).to(DEVICE)

# ── SublatticeLMHead (replaces default head — DegEq-immune + Zipf hierarchy) ─
try:
    _substrate = student.core_model.encoder.layers[0].attention.substrate
    _sublattice_head = SublatticeLMHead(
        n_embd             = CFG["model"]["n_embd"],
        vocab_size         = CFG["model"]["vocab_size"],
        substrate          = _substrate,
        r_max_frequent     = 1.5,    # frequent tokens near origin
        r_max_rare         = 5.0,    # rare tokens near boundary (Zipf)
        frequency_threshold= 0.3,
        learnable_scale    = True,
        angular_mode       = True,   # ∂logit/∂r = 0 — Channel 2 DegEq fix
    )
    student.core_model.lm_head = _sublattice_head
    student = student.to(DEVICE)     # remount after head swap — fixes CPU/CUDA mismatch
    print(f"  lm_head: SublatticeLMHead installed ✅")
    print(f"    Sublattice A (frequent): r_max=1.5  |  Sublattice B (rare): r_max=5.0")
    print(f"    angular_mode=True → ∂logit/∂r = 0 (DegEq immune)")
except Exception as _sl_e:
    print(f"  ⚠️  SublatticeLMHead failed: {_sl_e}")
    print(f"  Falling back to AngularLMHeadV2...")
    _sub = student.core_model.substrate
    student.core_model.lm_head = AngularLMHeadV2(
        n_embd       = CFG["model"]["n_embd"],
        vocab_size   = VOCAB_SIZE,
        substrate    = _sub,
        temperature  = 20.0,
        learnable_temp = False,
    ).to(DEVICE)
    print(f"  lm_head: AngularLMHeadV2 (fallback)")

n_params = student.num_parameters()
_head_name = type(student.core_model.lm_head).__name__
print(f"\nStudent: {n_params:,} params ({n_params/1e6:.1f}M)")
print(f"  arch    : {student_cfg.n_layer}L × {student_cfg.n_embd}d × {student_cfg.n_head}H")
print(f"  dynamics: use_dynamics={student_cfg.use_dynamics}  K={student_cfg.coupling_strength}  dt={student_cfg.dt}")
print(f"  lm_head : {_head_name}")
print(f"  teacher/student ratio: {117e6/n_params:.1f}x")
print("✅ Models ready")

Tokenizer: loaded from Drive cache
Tokenizer: vocab=50257
Teacher: loading from Drive cache...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Teacher: GPT-2-small 768d [frozen] — from cache
  lm_head: SublatticeLMHead installed ✅
    Sublattice A (frequent): r_max=1.5  |  Sublattice B (rare): r_max=5.0
    angular_mode=True → ∂logit/∂r = 0 (DegEq immune)

Student: 63,563,105 params (63.6M)
  arch    : 12L × 512d × 8H
  dynamics: use_dynamics=True  K=0.3  dt=0.05
  lm_head : SublatticeLMHead
  teacher/student ratio: 1.8x
✅ Models ready


Cell 6 — OpenWebText DataLoader  [HARDENED: Drive cache → fallback download]
=============================================================================
Estratégia de cache em 3 camadas:
  1. Drive cache (HF Arrow format) → carrega em < 2 min se já baixou antes
  2. /content cache local (sessão ainda viva) → instantâneo
  3. Download fresco → só se as duas camadas acima falharem (~10-30 min)

Variável de controle:
  FORCE_REDOWNLOAD = True   → ignora cache e baixa tudo de novo (use só se corrompeu)
  FORCE_REDOWNLOAD = False  → comportamento normal (padrão)

In [5]:
import os
from pathlib import Path

FORCE_REDOWNLOAD = False  # ← mude para True só se o cache estiver corrompido

# ── Diretórios de cache ───────────────────────────────────────────────────
_HF_DRIVE_CACHE  = PAPER_ROOT / "hf_cache"          # cache persistente no Drive
_HF_LOCAL_CACHE  = Path("/content/hf_cache_local")  # cache rápido na sessão

_HF_DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
_HF_LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# Aponta HF para o cache do Drive como primário
os.environ["HF_DATASETS_CACHE"] = str(_HF_DRIVE_CACHE)
os.environ["HF_HOME"]           = str(_HF_DRIVE_CACHE)

if FORCE_REDOWNLOAD:
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    print("⚠️  FORCE_REDOWNLOAD=True — ignorando cache, baixando tudo de novo")
else:
    # Tenta offline primeiro; se falhar, libera download
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    print(f"📦 Cache Drive : {_HF_DRIVE_CACHE}")
    print(f"📦 Cache local : {_HF_LOCAL_CACHE}")

from cgt.distillation.dataset_v2 import build_openwebtext_loaders

def _build_loaders(offline: bool):
    os.environ["HF_DATASETS_OFFLINE"] = "1" if offline else "0"
    return build_openwebtext_loaders(
        tokenizer,
        seq_len           = CFG["model"]["n_positions"],
        batch_size        = CFG["training"]["batch_size"],
        overlap           = 64,
        num_workers       = 2,
        max_train_samples = 500_000,
    )

train_loader = val_loader = None

# Camada 1: tenta carregar do cache (modo offline)
if not FORCE_REDOWNLOAD:
    try:
        print("🔍 Tentando carregar do cache Drive (modo offline)...")
        train_loader, val_loader = _build_loaders(offline=True)
        print("✅ Dataset carregado do cache Drive — sem download!")
    except Exception as _cache_err:
        print(f"⚠️  Cache offline falhou: {type(_cache_err).__name__}: {_cache_err}")
        print("   → Ativando download...")

# Camada 2/3: download se necessário
if train_loader is None:
    try:
        print("🌐 Baixando OpenWebText (~12GB comprimido, 10-30min na primeira vez)...")
        train_loader, val_loader = _build_loaders(offline=False)
        print("✅ Download concluído — cache salvo em Drive para próxima sessão")
    except Exception as _dl_err:
        raise RuntimeError(
            f"Falha ao carregar dataset: {_dl_err}\n"
            f"Verifique conexão e espaço no Drive ({_HF_DRIVE_CACHE})"
        ) from _dl_err

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Val   batches : {len(val_loader):,}")
print(f"Tokens/step   : {CFG['training']['batch_size'] * CFG['model']['n_positions']:,}")
print(f"Cache Drive   : {_HF_DRIVE_CACHE}")
print("✅ Dataset ready")

📦 Cache Drive : /content/drive/MyDrive/HydraPaper_Magnetic/hf_cache
📦 Cache local : /content/hf_cache_local
🔍 Tentando carregar do cache Drive (modo offline)...
  [OpenWebText] Loading dataset...
  [OpenWebText] Cache hit: /content/drive/MyDrive/HydraPaper_VariantF/data/openwebtext


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1217 > 1024). Running this sequence through the model will result in indexing errors


  [OpenWebText] Capped to 500,000 samples
  [OpenWebText] 500,000 documents
  [OpenWebText] Tokenising 500,000 docs in-memory...
  [OpenWebText]   50,000/500,000  tokens so far: 56,737,332
  [OpenWebText]   100,000/500,000  tokens so far: 112,963,538
  [OpenWebText]   150,000/500,000  tokens so far: 170,126,390
  [OpenWebText]   200,000/500,000  tokens so far: 226,867,891
  [OpenWebText]   250,000/500,000  tokens so far: 283,174,962
  [OpenWebText]   300,000/500,000  tokens so far: 340,394,624
  [OpenWebText]   350,000/500,000  tokens so far: 397,322,913
  [OpenWebText]   400,000/500,000  tokens so far: 453,973,661
  [OpenWebText]   450,000/500,000  tokens so far: 510,056,246
  [OpenWebText]   500,000/500,000  tokens so far: 566,334,506
  [OpenWebText] 1,200,932 train / 63,206 val windows
✅ Dataset carregado do cache Drive — sem download!

Train batches : 150,117
Val   batches : 7,901
Tokens/step   : 4,096
Cache Drive   : /content/drive/MyDrive/HydraPaper_Magnetic/hf_cache
✅ Dataset re

In [6]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch.cuda.empty_cache()

Cell 7 — Competitive Training

In [ ]:
import time, csv, torch
from pathlib import Path
from cgt.distillation import DistillationConfigV2, DistillationTrainerV2, RadiusCollapseAbort
# ── Gemini modules: monitoring + geometry ──────────────────────────────
from cgt.diagnostics.hpc_guard   import HPCTrainingGuard
from cgt.diagnostics.hysteresis  import HysteresisDetector
from cgt.losses.volume_weighted  import VolumeWeightedCE
from cgt.dynamics.pcgrad         import PCGrad
from cgt.dynamics.geometric_controller import GeometricController, ControllerConfig
from cgt.integration.magnetic_friction import MagneticFrictionExtension, MagneticFrictionConfig
from cgt.dynamics.magnetic_friction_scheduler import FrictionAwareCouplingScheduler
from cgt.integration.magnetic_friction import MagneticFrictionExtension, MagneticFrictionConfig
from cgt.dynamics.magnetic_friction_scheduler import FrictionAwareCouplingScheduler


RUN_TAG   = "hydra_magnetic_v1"
MAX_STEPS = CFG["training"]["max_steps"]

_log_dir  = PAPER_ROOT / "logs"        / RUN_TAG
_ckpt_dir = PAPER_ROOT / "checkpoints" / RUN_TAG
_log_dir.mkdir(parents=True, exist_ok=True)
_ckpt_dir.mkdir(parents=True, exist_ok=True)
print(f"Run → {_log_dir}")

# ── Config ────────────────────────────────────────────────────────────────
dist_cfg = DistillationConfigV2(
    alpha                    = CFG["training"]["alpha"],
    temperature              = CFG["training"]["temperature"],
    lambda_distill           = CFG["training"]["lambda_distill"],
    lambda_hidden            = CFG["training"]["lambda_hidden"],
    lambda_radius            = CFG["training"]["lambda_radius"],
    lambda_contrast          = CFG["training"]["lambda_contrast"],
    label_smoothing          = CFG["training"]["label_smoothing"],
    learning_rate            = CFG["training"]["learning_rate"],
    weight_decay             = CFG["training"]["weight_decay"],
    max_steps                = MAX_STEPS,
    warmup_steps             = CFG["training"]["warmup_steps"],
    gradient_clip            = CFG["training"]["gradient_clip"],
    eval_every               = CFG["training"]["eval_every"],
    log_every                = CFG["training"]["log_every"],
    checkpoint_every         = CFG["training"]["checkpoint_every"],
    lr_floor                 = CFG["training"]["lr_floor"],
    teacher_hidden_dim       = CFG["training"]["teacher_hidden_dim"],
    riemannian_correct_vocab   = True,
    riemannian_correct_embed   = True,
    riemannian_correct_encoder = True,
    use_oted                 = True,
    oted_r_target            = 1.5,
    use_angular_head         = False,
    adaptive_lambda          = False,
    deg_eq_action            = "none",  # desabilita AutoTune
    early_stopping_patience  = CFG["early_stopping"]["patience"],
    early_stopping_min_delta = CFG["early_stopping"]["min_delta"],
    ema_beta                 = CFG["early_stopping"]["ema_beta"],
    ema_beta_fast            = CFG["early_stopping"]["ema_beta_fast"],
    min_steps_before_stopping= CFG["early_stopping"]["min_steps"],
    trend_window             = CFG["early_stopping"]["window_size"],
    noise_mult               = CFG["early_stopping"]["noise_mult"],
    plateau_threshold        = CFG["early_stopping"]["plateau_threshold"],
    logit_std_min_delta      = CFG["early_stopping"]["logit_std_delta"],
    keep_last_n_checkpoints  = 3,
    radius_collapse_abort    = True,
)

# ── Reset lambdas + temperature (belt & suspenders) ──────────────────────
dist_cfg.lambda_distill = CFG["training"]["lambda_distill"]  # 0.01
dist_cfg.lambda_hidden  = CFG["training"]["lambda_hidden"]   # 0.01
dist_cfg.temperature    = 1.0   # reset — AutoTune may have raised to 1.44
dist_cfg.deg_eq_action  = "none"

# ── Resume check ──────────────────────────────────────────────────────────
_best_ckpt = _ckpt_dir / "distill_v2_best.pt"
_resume    = _best_ckpt.exists()
print(f"  {'🔄 Resuming' if _resume else '🆕 Starting'} — {_best_ckpt.name if _resume else 'from scratch'}")

# ── Memory setup ─────────────────────────────────────────────────────────
import torch as _tc
_tc.cuda.empty_cache()
_tc.set_float32_matmul_precision("high")
print("  ℹ️  Running in eager mode (torch.compile disabled)")

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = DistillationTrainerV2(
    student        = student,
    teacher        = teacher,
    config         = dist_cfg,
    tokenizer      = tokenizer,
    checkpoint_dir = _ckpt_dir,
    device         = DEVICE,
)

if _resume:
    try:
        trainer.load_checkpoint(_best_ckpt)
        print(f"  ✅ Resumed at step {trainer.step}  best_val={trainer.best_val:.4f}")
        # ── Force overrides AFTER load (checkpoint restores old config) ──────
        trainer.config.deg_eq_action  = "none"  # disable AutoTune permanently
        trainer.config.lambda_distill = CFG["training"]["lambda_distill"]  # 0.01
        trainer.config.lambda_hidden  = CFG["training"]["lambda_hidden"]   # 0.01
        trainer.config.temperature    = 1.0     # reset temperature (AutoTune may have raised it)
        print(f"  ✅ Post-load override: deg_eq_action=none, λ_kl={trainer.config.lambda_distill}, λ_hid={trainer.config.lambda_hidden}, T={trainer.config.temperature}")
    except Exception as e:
        print(f"  ⚠️  Resume failed: {e} — starting fresh")

# ── AdaptiveHyperController (depois do trainer) ───────────────────────────
try:
    from cgt.distillation import AdaptiveHyperController, AdaptiveControllerConfig
    controller = AdaptiveHyperController(
        config = AdaptiveControllerConfig(
            grad_balance_every = 500,    # CGT: faster correction
            balance_tolerance  = 2.0,
            balance_lr         = 0.5,
            ppl_stall_patience = 5,
            lr_reduce_factor   = 0.7,
            regime_every       = 200,
            rdc_danger         = 3.0,
            logit_std_max      = 15.0,
            verbose            = True,
        ),
        trainer = trainer,
    )
    trainer.adaptive_controller = controller
    print(f"  [AdaptiveController] ✅ Activo  (L1@500  rad_max=4.0)")
except ImportError:
    print("  [AdaptiveController] ⚠️  não disponível")
    class _Stub:
        cfg = type('cfg', (), {'grad_balance_every': 5000, 'regime_every': 200})()
        def summary(self): return "controller não carregado"
    controller = _Stub()

# ── Gemini Step 1: HPCGuard + HysteresisDetector (monitoring) ───────────
hpc_guard = HPCTrainingGuard(
    patience=500,
    abort_rdc_threshold=8.0,
)
hysteresis_det = HysteresisDetector(window_size=200)
print("  [Gemini] ✅ HPCGuard + HysteresisDetector active")

# ── Gemini Step 2: VolumeWeightedCE (geometric loss, strength=0.1) ─────────
try:
    _sub = trainer._get_substrate(student) if hasattr(trainer, "_get_substrate") else None
    vol_ce = VolumeWeightedCE(substrate=_sub, K=1.0, strength=0.1)
    print("  [Gemini] ✅ VolumeWeightedCE active (blend=0.1)")
except Exception as _ve:
    vol_ce = None
    print(f"  [Gemini] ⚠️  VolumeWeightedCE: {_ve}")

# ── Gemini Step 3: PCGrad (gradient surgery) ─────────────────────────────
try:
    pcgrad_inst = PCGrad(reduction="mean")
    print("  [Gemini] ✅ PCGrad active")
except Exception as _pe:
    pcgrad_inst = None
    print(f"  [Gemini] ⚠️  PCGrad: {_pe}")

# ── Gemini Step 4: GeometricController (closed-loop) ─────────────────────
try:
    geo_ctrl = GeometricController(
        loss_fn=trainer.loss_fn,
        config=ControllerConfig(
            rdc_safe=2.0,
            rdc_warning=5.0,
            rdc_critical=8.0,
            warmup_steps=500,
        ),
    )
    print("  [Gemini] ✅ GeometricController active")
except Exception as _ge:
    geo_ctrl = None
    print(f"  [Gemini] ⚠️  GeometricController: {_ge}")

# ── SGDR: warm restart only if resuming mid-training ────────────────────────
_cur_step = trainer.step
if _cur_step >= 50000:
    _restart_lr = 1.5e-4
    for _pg in trainer.optimizer.param_groups:
        _pg["lr"] = _restart_lr
    print(f"  [SGDR] ✅ Warm restart: LR → {_restart_lr:.2e} @ step {_cur_step}")
else:
    print(f"  [SGDR] Starting fresh from step {_cur_step} — no restart needed")

# ── Magnetic Step 5: FrictionAwareCouplingScheduler ─────────────────────
try:
    friction_scheduler = FrictionAwareCouplingScheduler(
        coupling_min=0.1, coupling_max=2.0, warmup_steps=1000,
        frustration_threshold=0.5, frustration_target=0.25,
        retreat_rate=0.02, advance_rate=0.005,
    )
    print("  [Magnetic] ✅ FrictionAwareCouplingScheduler active")
except Exception as _fs_e:
    friction_scheduler = None
    print(f"  [Magnetic] ⚠️  FrictionAwareCouplingScheduler: {_fs_e}")

# ── Magnetic Step 6: MagneticFrictionExtension ───────────────────────────
try:
    mag_cfg = MagneticFrictionConfig(
        enable_frustration_coupling   = True,
        enable_sublattice_lm_head     = True,
        enable_hysteresis_detection   = True,
        enable_phase_analysis         = True,
        enable_nonmonotonic_scheduler = True,
        frustration_strength          = 0.3,
        frustration_ceiling           = 0.7,
        r_max_frequent                = 1.5,
        r_max_rare                    = 5.0,
        coupling_min                  = 0.1,
        coupling_max                  = 2.0,
        warmup_steps                  = 1000,
    )
    mag_ext = MagneticFrictionExtension(
        config=mag_cfg, num_heads=CFG["model"]["n_head"],
    )
    print("  [Magnetic] ✅ MagneticFrictionExtension active")
except Exception as _me:
    mag_ext = None
    print(f"  [Magnetic] ⚠️  MagneticFrictionExtension: {_me}")

# ── Preflight ─────────────────────────────────────────────────────────────
_head = type(getattr(getattr(trainer.student,"core_model",trainer.student),"lm_head",None)).__name__
print(f"  lm_head  : {_head}")
print(f"  λ CE/KL/hid : {dist_cfg.alpha} / {dist_cfg.lambda_distill} / {dist_cfg.lambda_hidden}")
print(f"  steps    : {trainer.step} → {MAX_STEPS}")
print(f"  controller: L1@{controller.cfg.grad_balance_every}  L2@eval  L3@{controller.cfg.regime_every}")

# ── CSV flush ─────────────────────────────────────────────────────────────
import csv as _csv

def _flush(trainer, log_dir):
    """Persiste train_hist e val_hist em disco atomicamente."""
    for fname, hist in [("training_metrics.csv", trainer.train_hist),
                        ("val_metrics.csv",      trainer.val_hist)]:
        if not hist:
            continue
        tmp = log_dir / (fname + ".tmp")
        dst = log_dir / fname
        with open(tmp, "w", newline="") as f:
            w = _csv.DictWriter(f, fieldnames=list(hist[0].keys()), extrasaction="ignore")
            w.writeheader()
            w.writerows(hist)
        tmp.replace(dst)  # escrita atômica — nunca deixa CSV pela metade

# Mantém compatibilidade com chamadas antigas via trainer._flush_csv
trainer._flush_csv = lambda th, vh: _flush(trainer, _log_dir)

# ── Periodic background flush ──────────────────────────────────────────────
# trainer.train() é opaco (loop interno ao DistillationTrainerV2), então
# não dá para chamar flush dentro do step diretamente. A thread de background
# acessa trainer.train_hist / trainer.val_hist direto da memória a cada intervalo.
import threading

_FLUSH_INTERVAL = 300   # segundos entre cada flush automático (5 min)
_flush_stop     = threading.Event()

def _bg_flush_loop():
    while not _flush_stop.wait(_FLUSH_INTERVAL):
        try:
            _flush(trainer, _log_dir)
            _step = getattr(trainer, "step", "?")
            _nt   = len(getattr(trainer, "train_hist", []))
            _nv   = len(getattr(trainer, "val_hist",   []))
            print(f"  [auto-flush] ✅ step={_step}  train={_nt}  val={_nv}  → {_log_dir.name}/")
        except Exception as _fe:
            print(f"  [auto-flush] ⚠️  {type(_fe).__name__}: {_fe}")

_flush_thread = threading.Thread(target=_bg_flush_loop, daemon=True, name="csv-flusher")
_flush_thread.start()
print(f"  [auto-flush] Thread iniciada — intervalo={_FLUSH_INTERVAL}s ({_FLUSH_INTERVAL//60} min)")
print(f"  [auto-flush] CSVs em: {_log_dir}")

# ── Train ─────────────────────────────────────────────────────────────────
remaining = MAX_STEPS - trainer.step
print(f"\n  🚀 Training {remaining:,} steps...")
print(f"  Est. time on T4: ~{remaining/1000*4/60:.0f}h")
t0 = time.time()
try:
    train_hist, val_hist = trainer.train(train_loader, val_loader)
except RadiusCollapseAbort as e:
    print(f"  ⚡ RadiusCollapse: {e}")
    train_hist = getattr(trainer, "train_hist", [])
    val_hist   = getattr(trainer, "val_hist",  [])
finally:
    # Garante que a thread para e faz flush final independente de crash
    _flush_stop.set()
    _flush(trainer, _log_dir)
    print(f"  [auto-flush] 🛑 Thread encerrada + flush final executado")

elapsed = time.time() - t0
print(controller.summary())
# ── Gemini post-training telemetry ───────────────────────────────────────
if val_hist:
    for r in val_hist[-5:]:
        _rdc = float(r.get("rdc_ema", 0) or 0)
        _action = hpc_guard.step(rdc_ema=_rdc, step=int(r.get("step", 0)))
        hysteresis_det.update(rdc_ema=_rdc)
    print(f"  [HPCGuard] {hpc_guard.report()}")
    if hysteresis_det.hysteresis_detected():
        print(f"  [Hysteresis] ⚠️  Detected — cycles={hysteresis_det.get_cycle_count()}")
    if geo_ctrl is not None:
        last = val_hist[-1]
        geo_ctrl.step({"rdc_ema": float(last.get("rdc_ema", 0) or 0),
                       "val_ppl": float(last.get("val_ppl", 999)),
                       "step": int(last.get("step", 0))})
        print(f"  [GeoCtrl] {geo_ctrl.report()}")
    if mag_ext is not None:
        _rdc_last = float(val_hist[-1].get("rdc_ema", 0) or 0)
        mag_ext.step(rdc_ema=_rdc_last, step=val_steps[-1] if val_hist else 0)
        print(f"  [Magnetic] {mag_ext.report()}")
try:
    trainer.save(is_best=True)
    torch.save(student.state_dict(), _ckpt_dir / "student_competitive.pt")
    print(f"  💾 Saved → {_ckpt_dir}")
except Exception as e:
    print(f"  ⚠️  Save failed: {e}")

rdcs = [float(r["rdc_ema"]) for r in val_hist if r.get("rdc_ema") not in (None,"","nan")]
ppls = [float(r["val_ppl"]) for r in val_hist if r.get("val_ppl")]
rdc_star = round(sum(rdcs[-5:])/min(5,len(rdcs)),2) if rdcs else None
best_ppl = round(min(ppls),1) if ppls else None
print(f"\n  ✅ {elapsed/3600:.1f}h  rdc*={rdc_star}  best_ppl={best_ppl}")
if best_ppl and best_ppl <= 35:
    print("  🏆 SUPEROU GPT-2-small! (PPL ≤ 35)")
elif best_ppl and best_ppl <= 100:
    print("  ✅ Competitivo (PPL ≤ 100) — mais steps para superar")

# ── Cell: Métricas + Benchmark student vs teacher ────────────────────────────
import csv, math, time, torch
from pathlib import Path

_log      = PAPER_ROOT / "logs" / "hydra_competitive_v1"
_train_csv = _log / "training_metrics.csv"
_val_csv   = _log / "val_metrics.csv"

def read_csv(p):
    if not p.exists(): print(f"  ⚠️  {p.name} not found"); return []
    rows = list(csv.DictReader(open(p)))
    print(f"  {p.name}: {len(rows)} rows")
    return rows

train_rows = read_csv(_train_csv)
val_rows   = read_csv(_val_csv)

# ── Training summary ──────────────────────────────────────────────────────
if train_rows:
    steps  = [int(r["step"]) for r in train_rows]
    losses = [float(r["train_loss"]) for r in train_rows if r.get("train_loss") not in ("","nan")]
    ppls   = [float(r["train_ppl"])  for r in train_rows if r.get("train_ppl")  not in ("","nan","inf") and float(r["train_ppl"]) < 1e6]
    rdcs   = [float(r["rdc_proxy"])  for r in train_rows if r.get("rdc_proxy")  not in ("","nan")]
    print(f"\n── Training (steps {steps[0]}→{steps[-1]}) ──")
    print(f"  loss : {losses[0]:.3f} → {losses[-1]:.3f}  (best: {min(losses):.3f})")
    if ppls:  print(f"  ppl  : {ppls[0]:.1f} → {ppls[-1]:.1f}  (best: {min(ppls):.1f})")
    if rdcs:  print(f"  rdc  : {rdcs[0]:.2f} → {rdcs[-1]:.2f}  (best: {min(rdcs):.2f})")

# ── Validation summary ────────────────────────────────────────────────────
if val_rows:
    val_steps = [int(r["step"])      for r in val_rows]
    val_ppls  = [float(r["val_ppl"]) for r in val_rows if r.get("val_ppl") not in ("","nan")]
    val_rdcs  = [float(r["rdc_ema"]) for r in val_rows if r.get("rdc_ema") not in ("","nan")]
    best_ppl  = min(val_ppls)
    best_step = val_steps[val_ppls.index(best_ppl)]
    rdc_star  = round(sum(val_rdcs[-5:]) / min(5, len(val_rdcs)), 2) if val_rdcs else None
    print(f"\n── Validation ──")
    print(f"  best_ppl : {best_ppl:.2f}  @ step {best_step}")
    print(f"  last_ppl : {val_ppls[-1]:.2f}")
    print(f"  rdc*     : {rdc_star}  (< 2.0 = DegEq eliminado)")
    print(f"\n── Val history ──")
    print(f"  {'step':>7}  {'val_ppl':>9}  {'rdc_ema':>8}  {'phase':>6}")
    for r in val_rows:
        print(f"  {int(r['step']):>7}  {float(r.get('val_ppl',0)):>9.2f}  "
              f"{float(r.get('rdc_ema',0)):>8.3f}  {r.get('phase','?'):>6}")

# ── Live benchmark: student vs teacher ───────────────────────────────────
@torch.no_grad()
def eval_ppl(model, loader, device, max_batches=200, is_teacher=False):
    model.eval()
    total_loss = total_n = 0
    for i, batch in enumerate(loader):
        if i >= max_batches: break
        ids = batch["input_ids"].to(device)
        if is_teacher:
            out  = model.model(input_ids=ids, labels=ids)
            loss = out.loss; n = (ids != -100).sum().item()
        else:
            out  = model(input_ids=ids, labels=ids)
            loss = out.get("loss") or out.get("ce_loss"); n = ids.numel()
        if loss is not None and not torch.isnan(loss):
            total_loss += loss.item() * n; total_n += n
    avg = total_loss / max(total_n, 1)
    return math.exp(avg), avg

@torch.no_grad()
def bench_tps(model, seq, bs, device, n=20, is_teacher=False):
    dummy = torch.randint(0, 50257, (bs, seq), device=device)
    for _ in range(3):
        model.model(input_ids=dummy) if is_teacher else model(input_ids=dummy)
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        model.model(input_ids=dummy) if is_teacher else model(input_ids=dummy)
    torch.cuda.synchronize()
    return n * bs * seq / (time.time() - t0)

print("\nBenchmarking (aguarde ~2 min)...")
SEQ = CFG["model"]["n_positions"]
s_ppl, s_loss = eval_ppl(student, val_loader, DEVICE, is_teacher=False)
t_ppl, t_loss = eval_ppl(teacher,  val_loader, DEVICE, is_teacher=True)
s_tps = bench_tps(student, SEQ, 4, DEVICE, is_teacher=False)
t_tps = bench_tps(teacher,  SEQ, 4, DEVICE, is_teacher=True)
s_p   = sum(p.numel() for p in student.parameters())
t_p   = sum(p.numel() for p in teacher.model.parameters())

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║         HyDRA-Competitive vs GPT-2-small                    ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  {'Metric':<22} {'HyDRA':>15} {'GPT-2':>15} ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  {'Params':<22} {s_p/1e6:>13.1f}M {t_p/1e6:>13.1f}M ║")
print(f"║  {'Val PPL':<22} {s_ppl:>15.2f} {t_ppl:>15.2f} ║")
print(f"║  {'Val Loss':<22} {s_loss:>15.4f} {t_loss:>15.4f} ║")
print(f"║  {'Throughput (tok/s)':<22} {s_tps:>13,.0f} {t_tps:>13,.0f} ║")
print(f"║  {'Speed ratio':<22} {s_tps/t_tps:>14.2f}x {'1.00x':>15} ║")
print(f"║  {'PPL ratio':<22} {s_ppl/t_ppl:>14.2f}x {'1.00x':>15} ║")
print("╠══════════════════════════════════════════════════════════════╣")
ratio = s_ppl / t_ppl
if ratio <= 1.0:   print("║  🏆 SUPEROU O TEACHER em PPL!                               ║")
elif ratio <= 1.5: print(f"║  ✅ Competitivo: {ratio:.2f}x teacher PPL                        ║")
elif ratio <= 3.0: print(f"║  📈 Progredindo: {ratio:.2f}x — precisa mais steps               ║")
else:              print(f"║  🔄 Ainda longe: {ratio:.2f}x — modelo ainda convergindo          ║")
if s_tps > t_tps:  print(f"║  ⚡ {s_tps/t_tps:.1f}x mais rápido que o teacher                      ║")
print("╚══════════════════════════════════════════════════════════════╝")

Run → /content/drive/MyDrive/HydraPaper_Magnetic/logs/hydra_magnetic_v1
  🔄 Resuming — distill_v2_best.pt
  ℹ️  Running in eager mode (torch.compile disabled)
🔄 Loading checkpoint: /content/drive/MyDrive/HydraPaper_Magnetic/checkpoints/hydra_magnetic_v1/distill_v2_best.pt


/tmp/ipykernel_11091/3619015875.py:83: UserWarning: [Paper2] OTED active: Origin-Tangent Euclidean Distillation.
  trainer = DistillationTrainerV2(


✅ Loaded v2 checkpoint from step 1000
   ✅ Resumed at step=1000  best_val=6.2315
  ✅ Resumed at step 1000  best_val=6.2315
  ✅ Post-load override: deg_eq_action=none, λ_kl=0.01, λ_hid=0.05, T=1.0
  [AdaptiveController] Initialised — mode=single-GPU  L1@500  L2@every_eval  L3@200
  [AdaptiveController] ✅ Activo  (L1@500  rad_max=4.0)
  [Gemini] ✅ HPCGuard + HysteresisDetector active
  [Gemini] ✅ VolumeWeightedCE active (blend=0.1)
  [Gemini] ✅ PCGrad active
  [Gemini] ✅ GeometricController active
  [SGDR] Starting fresh from step 1000 — no restart needed
  [Magnetic] ✅ FrictionAwareCouplingScheduler active
  [Magnetic] ✅ MagneticFrictionExtension active
  lm_head  : SublatticeLMHead
  λ CE/KL/hid : 1.0 / 0.01 / 0.05
  steps    : 1000 → 200000
  controller: L1@500  L2@eval  L3@200
  [auto-flush] Thread iniciada — intervalo=300s (5 min)
  [auto-flush] CSVs em: /content/drive/MyDrive/HydraPaper_Magnetic/logs/hydra_magnetic_v1

  🚀 Training 199,000 steps...
  Est. time on T4: ~13h

🎓 STARTI

Cell 8 — Chat with trained model

In [ ]:
import torch
import torch.nn.functional as F

MAX_NEW_TOKENS = 150
TEMPERATURE    = 0.7
TOP_K          = 50
TOP_P          = 0.9
REPETITION_PEN = 1.3

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
             rep_penalty=REPETITION_PEN):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    gen = ids.clone()
    for _ in range(max_new_tokens):
        ctx    = gen[:, -CFG["model"]["n_positions"]:]
        logits = model(ctx)["logits"][:, -1, :]
        if temperature != 1.0: logits = logits / max(temperature, 1e-6)
        if rep_penalty != 1.0:
            for t in set(gen[0].tolist()):
                logits[0,t] = logits[0,t]/rep_penalty if logits[0,t]>0 else logits[0,t]*rep_penalty
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
        if top_p < 1.0:
            s_logits, s_idx = torch.sort(logits, descending=True)
            cum = s_logits.softmax(-1).cumsum(-1)
            s_logits[cum - s_logits.softmax(-1) > top_p] = float("-inf")
            logits = torch.zeros_like(logits).scatter_(1, s_idx, s_logits)
        next_tok = torch.multinomial(F.softmax(logits, -1), 1)
        gen = torch.cat([gen, next_tok], dim=1)
        if next_tok.item() == tokenizer.eos_token_id: break
    return tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)

# ── Load best checkpoint ──────────────────────────────────────────────────
_ckpt = _ckpt_dir / "student_competitive.pt"
if _ckpt.exists():
    student.load_state_dict(torch.load(str(_ckpt), map_location=DEVICE))
    student.eval()
    print(f"✅ Loaded {_ckpt.name}")

print("╔══════════════════════════════════════════════════════╗")
print("║         HyDRA-Competitive Chat  (H⁵¹², K=1)        ║")
print("║   64M params · OpenWebText · AngularLMHeadV2        ║")
print("╚══════════════════════════════════════════════════════╝")
print("Commands: /temp /topk /topp /tokens /rep /quit\n")

gen_cfg = dict(temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
               max_new_tokens=MAX_NEW_TOKENS, rep_penalty=REPETITION_PEN)
context = ""
while True:
    try: user = input("You: ").strip()
    except (EOFError, KeyboardInterrupt): print("\n[end]"); break
    if not user: continue
    if user.startswith("/"):
        p = user.split()
        try:
            if p[0]=="/quit": break
            elif p[0]=="/temp":   gen_cfg["temperature"]    = float(p[1])
            elif p[0]=="/topk":   gen_cfg["top_k"]          = int(p[1])
            elif p[0]=="/topp":   gen_cfg["top_p"]          = float(p[1])
            elif p[0]=="/tokens": gen_cfg["max_new_tokens"] = int(p[1])
            elif p[0]=="/rep":    gen_cfg["rep_penalty"]    = float(p[1])
            print(f"  {p[0][1:]} = {p[1] if len(p)>1 else "?"}")
        except: print(f"  usage: {p[0]} <value>")
        continue
    resp = generate(student, tokenizer, context + user + " ", **gen_cfg)
    print(f"HyDRA: {resp}\n")
    context = (context + user + " " + resp + " ")[-400:]